# 7. Learn and evaluate a baseline

**Estimated session:** 30–45 minutes, plus time to write your interpretation.

| Before running | Your task |
|---|---|
| Input | `outputs/checkpoints/farm-split.rds` from Chapter 6 |
| Working file | Open `scripts/07-baseline.R`; Save As `scripts/my-07-baseline.R` |
| Starting state | Restart R; reload the split in the first block |
| Stop after | Saving audit counts and completing the final checkpoint |
| Finish with | `outputs/baseline/scores.csv`, predictions, plot and a short interpretation |

## 7.1 Load the saved roles

In [1]:
if (!file.exists("outputs/checkpoints/farm-split.rds")) stop("Run Chapter 6 first to create farm-split.rds.")
split_checkpoint <- readRDS("outputs/checkpoints/farm-split.rds")
train <- split_checkpoint$train
test <- split_checkpoint$test
farm_ids <- split_checkpoint$farm_ids
test_farms <- split_checkpoint$test_farms

The container restores the tables called train and test. We will learn from train and evaluate on test. No object from an earlier R session is required.

## 7.2 Fit a rule you can explain


Use **only the training outcomes** to calculate a mean. Every test record receives that same prediction. No weather or management predictors are used in this baseline.

In [2]:
baseline_log10 <- mean(train$log10_msc)
baseline_log10
test$prediction_log10 <- rep(baseline_log10, nrow(test))
stopifnot(all(test$prediction_log10 == mean(train$log10_msc)))

[1] 0.7450425

The prediction is about 0.7450 log10 CFU/mL for every held-out record. Do not calculate the baseline from all 525 outcomes: that would use the test outcomes in the model.


The fitted model here is this one estimated mean, not a set of weather or management coefficients. Every held-out sample receives it. It is useful as a reference against which later procedures can be compared.

## 7.3 Calculate and interpret error


A residual here is **observed minus predicted**. RMSE is the square root of the average squared residual; MAE is the average absolute residual. Both are reported on the log10 CFU/mL scale, and smaller values mean less error.

In [3]:
test$error_log10 <- test$log10_msc - test$prediction_log10
rmse <- sqrt(mean(test$error_log10^2))
mae <- mean(abs(test$error_log10))
data.frame(model = "Training-mean baseline", n_test = nrow(test),
           RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae)

model,n_test,RMSE_log10_CFU_ml,MAE_log10_CFU_ml
<chr>,<int>,<dbl>,<dbl>
Training-mean baseline,105,0.5289231,0.409148


These are record-weighted errors on this one held-out set. They are not confidence intervals and do not establish which later model will perform best. This split is used here for a fixed baseline demonstration; later model tuning must use the training portion rather than repeatedly consulting these test results.


Return to the two-row hand table in [Chapter 4](04-small-farm-example.ipynb) if the formulas feel abstract. Here the observed and predicted values are both on the log10 scale. An error of 1 on that scale corresponds to a tenfold concentration ratio; the signs show which value was larger. RMSE 0.529 is **not a 52.9% error rate**, and it should not be described as a typical fold error without further definition.

We have established a starting score on one fixed split. Judging whether it is useful requires the intended application, acceptable errors and comparisons evaluated using an appropriate validation procedure. The current pilot does not establish an application-ready predictor.

## 7.4 Reconstruct before accepting

In [4]:
rmse_check <- sqrt(sum((test$log10_msc - baseline_log10)^2) / nrow(test))
mae_check <- sum(abs(test$log10_msc - baseline_log10)) / nrow(test)
stopifnot(isTRUE(all.equal(rmse, rmse_check)))
stopifnot(isTRUE(all.equal(mae, mae_check)))
round(data.frame(observed = head(test$log10_msc, 3),
                 predicted = head(test$prediction_log10, 3),
                 residual = head(test$error_log10, 3)), 4)

observed,predicted,residual
<dbl>,<dbl>,<dbl>
0.8129,0.745,0.0679
0.5441,0.745,-0.2010
-0.3010,0.745,-1.0461


Use the first displayed row to subtract predicted from observed yourself. The table is rounded for display; the assertions use full precision. The second expressions for RMSE and MAE check the calculation over every test row.

**Ask AI:**

> I split my milk records by FarmID before calculating the training mean of log10 MSC. Write a short base-R check that no farms occur in both train and test, and reconstruct RMSE from the saved observations and predictions. Do not refit the model or change the split. Explain what this check does not establish.

Compare the proposed check with the actual objects and output above. An assistant must not invent a better score or claim that the split proves independence.


## 7.5 Keep the detection-limit caveat visible


The main score includes the encoded below-detection outcomes. As a descriptive sensitivity check, evaluate the **same fixed predictions** on test rows not carrying that flag:

In [5]:
# Evaluation subset only: the baseline is not refitted or chosen using this result.
quantified_test <- test[!test$left_censored, ]
quantified_rmse <- sqrt(mean(quantified_test$error_log10^2))
data.frame(evaluation = c("All eligible test records", "Quantified test records only"),
           n = c(nrow(test), nrow(quantified_test)),
           RMSE_log10_CFU_ml = c(rmse, quantified_rmse))

evaluation,n,RMSE_log10_CFU_ml
<chr>,<int>,<dbl>
All eligible test records,105,0.5289231
Quantified test records only,104,0.5062914


Only one test record has the flag in this split. The second score describes a different subset. It does not estimate error for all unobserved true concentrations, and it does not fix the censoring problem. A later study would examine more suitable censored-data methods and sensitivity analyses.


## 7.6 Save the evidence

In [6]:
dir.create("outputs/baseline", recursive = TRUE, showWarnings = FALSE)
score <- data.frame(model = "Training-mean baseline", training_mean_log10 = baseline_log10,
                    train_rows = nrow(train), test_rows = nrow(test),
                    train_farms = length(unique(train$FarmID)), test_farms = length(test_farms),
                    RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae,
                    quantified_test_n = nrow(quantified_test), quantified_test_RMSE = quantified_rmse)
write.csv(score, "outputs/baseline/scores.csv", row.names = FALSE)
split_table <- data.frame(FarmID = farm_ids,
                         split = ifelse(farm_ids %in% test_farms, "test", "train"))
write.csv(split_table, "outputs/baseline/farm-split.csv", row.names = FALSE)
write.csv(test[c("FarmID", "Sampling", "left_censored", "log10_msc", "prediction_log10", "error_log10")],
          "outputs/baseline/test-predictions.csv", row.names = FALSE)
writeLines(capture.output(sessionInfo()), "outputs/baseline/session-info.txt")
file.exists("outputs/baseline/scores.csv")

[1] TRUE

Open **Files → outputs → baseline**. You should see:

- `scores.csv`: the fitted training mean and evaluation metrics;
- `farm-split.csv`: the saved group assignment;
- `test-predictions.csv`: the held-out observations, predictions and residuals;
- `session-info.txt`: the R environment used for this run.

Create a plot using the training data:

In [7]:
png("outputs/baseline/training-distribution.png", width = 1200, height = 720, res = 150)
par(mar = c(4.5, 4.5, 2.5, 1), family = "sans", col.axis = "#294943", col.lab = "#294943")
hist(train$log10_msc, breaks = 18, col = "#8cc7b6", border = "white",
     main = "The baseline is learned from training farms",
     xlab = "Training log10 MSC (CFU/mL)", ylab = "Number of records")
abline(v = baseline_log10, col = "#a94c31", lwd = 3)
legend("topright", "Training mean", col = "#a94c31", lwd = 3, bty = "n")
invisible(dev.off())
cat("Saved outputs/baseline/training-distribution.png\n")

Saved outputs/baseline/training-distribution.png


![Distribution of training log10 MSC with the training mean marked by a vertical line.](../outputs/baseline/training-distribution.png)

Export the selection and verification counts:

In [8]:
write.csv(split_checkpoint$audit, "outputs/baseline/data-audit.csv", row.names = FALSE)
cat("All data and baseline checks passed.\n")

All data and baseline checks passed.


## 7.7 Write a result that matches your run


> Using the prespecified farm-level split, a training-mean baseline predicted held-out log10 MSC with RMSE about 0.529 and MAE about 0.409 log10 CFU/mL. The model used 420 training records and was evaluated on 105 records from 21 other farms. It used no management or weather predictors. Its score depends on this split and the treatment of below-detection values. The result does not establish causality or product safety.

The figures above come from executed code. When your input, split or target changes, regenerate the results and update the text. Do not reuse a paragraph containing stale numbers.

```{admonition} Final checkpoint
:class: tip
Find your my-07 script and saved outputs. Explain the input checkpoint and the training-only mean. Then restart R and run a my- copy of `scripts/worked-example.R` from top to bottom to reconstruct the whole real-data route. Show the zero-overlap result, reconstruct one residual, locate the saved score, and explain why the baseline mean came from training data only.
```

**Transfer exercise:** in a separate copy, use a different seed. Before running, predict which outputs can change and which scientific rules must remain true. Record the original and revised settings; do not pick a seed because it produces a preferred score.

[Download the full R script](../scripts/worked-example.R). The [downloads page](../reference/downloads.md) also contains the complete student project.

```{admonition} Open after you try: reference interpretation
:class: tip dropdown
The supplied split gives RMSE about 0.5289 and MAE about 0.4091 log10 CFU/mL on 105 records from 21 held-out farms. These are errors on the transformed target, not accuracy percentages, confidence intervals, causal effects or milk-safety decisions. The 0.125 source encoding remains a target limitation. A single score does not establish which later model will perform best.
```